In [1]:

import numpy as np
from scipy.signal import welch


def extract_advanced_features(signal, prefix):
    """
    Extract 16 features from one sensor signal.

    Parameters
    ----------
    signal : array-like
        Raw sensor signal.
    prefix : str
        Channel prefix, e.g. vibration_1 or phase_current_1.

    Returns
    -------
    dict
        Feature names and numerical values.
    """

    signal = np.asarray(signal, dtype=np.float64).ravel()

    # Remove invalid values
    signal = signal[np.isfinite(signal)]

    if len(signal) < 2:
        raise ValueError(
            "Signal must contain at least two finite values."
        )

    # Center the signal
    centered = signal - np.mean(signal)
    absolute_signal = np.abs(centered)

    # Basic time-domain quantities
    mean_abs = np.mean(absolute_signal)
    rms = np.sqrt(np.mean(centered ** 2))
    peak = np.max(absolute_signal)
    variance = np.var(centered)
    mean_square = np.mean(centered ** 2)

    epsilon = 1e-12

    # Time-domain features
    crest_factor = peak / (rms + epsilon)
    shape_factor = rms / (mean_abs + epsilon)
    impulse_factor = peak / (mean_abs + epsilon)
    peak_amplitude = peak
    zero_crossing_rate = np.mean(
        np.diff(np.signbit(centered)) != 0
    )

    # Welch power spectral density
    frequencies, power = welch(
        centered,
        fs=1.0,
        nperseg=min(2048, len(centered))
    )

    total_power = np.sum(power)
    normalized_power = power / (total_power + epsilon)

    spectral_centroid = np.sum(
        frequencies * normalized_power
    )

    spectral_entropy = -np.sum(
        normalized_power
        * np.log2(normalized_power + epsilon)
    )

    spectral_flatness = np.exp(
        np.mean(np.log(power + epsilon))
    ) / (np.mean(power) + epsilon)

    cutoff = np.median(frequencies)
    high_band_power_ratio = (
        np.sum(power[frequencies > cutoff])
        / (total_power + epsilon)
    )

    # Feature dictionary
    features = {
        f"{prefix}_crest_factor": crest_factor,
        f"{prefix}_shape_factor": shape_factor,
        f"{prefix}_impulse_factor": impulse_factor,
        f"{prefix}_peak_amplitude": peak_amplitude,
        f"{prefix}_variance": variance,
        f"{prefix}_zero_crossing_rate": zero_crossing_rate,
        f"{prefix}_spectral_centroid": spectral_centroid,
        f"{prefix}_spectral_entropy": spectral_entropy,
        f"{prefix}_spectral_flatness": spectral_flatness,
        f"{prefix}_high_band_power_ratio": high_band_power_ratio,
    }

    return features


In [4]:
import os

print("Current folder:", os.getcwd())
print("Files and folders:", os.listdir())

Current folder: C:\Users\aryan\Desktop\MachineFailurePredictor\src
Files and folders: ['.ipynb_checkpoints', 'feature_extraction.ipynb', 'inspect_data.ipynb']


In [7]:
from feature_extraction import extract_advanced_features

print("Feature extraction function imported successfully!")

Feature extraction function imported successfully!


In [8]:
import numpy as np
from feature_extraction import extract_advanced_features

# Create a sample vibration signal
sample_signal = np.random.randn(1000)

features = extract_advanced_features(sample_signal, "sensor1")

print("Number of features:", len(features))
print("Feature names:")
for name in features:
    print(name)

Number of features: 10
Feature names:
sensor1_crest_factor
sensor1_shape_factor
sensor1_impulse_factor
sensor1_peak_amplitude
sensor1_variance
sensor1_zero_crossing_rate
sensor1_spectral_centroid
sensor1_spectral_entropy
sensor1_spectral_flatness
sensor1_spectral_high_band_ratio


In [9]:
import os

print("Current folder:", os.getcwd())
print("\nFiles in this folder:")

for file in os.listdir():
    print(file)

Current folder: C:\Users\aryan\Desktop\MachineFailurePredictor\src

Files in this folder:
.ipynb_checkpoints
feature_extraction.ipynb
feature_extraction.py
inspect_data.ipynb
predictor.py
__pycache__


In [10]:
from pathlib import Path

project_path = Path(r"C:\Users\aryan\Desktop\MachineFailurePredictor")

print("Project folders and files:")

for item in project_path.iterdir():
    print(item.name)

Project folders and files:
data
firmware
models
notebooks
reports
src


In [11]:
from pathlib import Path

project_path = Path(r"C:\Users\aryan\Desktop\MachineFailurePredictor")
data_path = project_path / "data"

for item in data_path.rglob("*"):
    if item.is_dir():
        print("FOLDER:", item.relative_to(project_path))
    elif item.suffix.lower() == ".mat":
        print("MAT FILE:", item.relative_to(project_path))

FOLDER: data\processed
FOLDER: data\raw
FOLDER: data\raw\paderborn
FOLDER: data\raw\paderborn\K001
FOLDER: data\raw\paderborn\K002
FOLDER: data\raw\paderborn\K003
FOLDER: data\raw\paderborn\K004
FOLDER: data\raw\paderborn\KA01
FOLDER: data\raw\paderborn\KA03
FOLDER: data\raw\paderborn\KA04
FOLDER: data\raw\paderborn\KB23
FOLDER: data\raw\paderborn\KI01
FOLDER: data\raw\paderborn\KI03
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_1.mat
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_10.mat
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_11.mat
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_12.mat
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_13.mat
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_14.mat
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_15.mat
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_16.mat
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_17.mat
MAT FILE: data\raw\paderborn\K001\N09_M07_F10_K001_18.mat
MAT FILE: data\raw\paderborn\K001\

In [12]:
from scipy.io import loadmat
from pathlib import Path

project_path = Path(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor"
)

mat_path = (
    project_path
    / "data"
    / "raw"
    / "paderborn"
    / "K001"
    / "N09_M07_F10_K001_1.mat"
)

mat_data = loadmat(mat_path)

print("Variables found in the recording:\n")

for name, value in mat_data.items():
    if not name.startswith("__"):
        print(name, "-> shape:", getattr(value, "shape", None))

Variables found in the recording:

N09_M07_F10_K001_1 -> shape: (1, 1)


In [13]:
from scipy.io import loadmat
from pathlib import Path

project_path = Path(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor"
)

mat_path = (
    project_path
    / "data"
    / "raw"
    / "paderborn"
    / "K001"
    / "N09_M07_F10_K001_1.mat"
)

mat_data = loadmat(mat_path)

recording = mat_data["N09_M07_F10_K001_1"]

print("Recording type:", type(recording))
print("Recording dtype:", recording.dtype)
print("Recording shape:", recording.shape)

print("\nStructure field names:")

for name in recording.dtype.names:
    print(name)

Recording type: <class 'numpy.ndarray'>
Recording dtype: [('Info', 'O'), ('X', 'O'), ('Y', 'O'), ('Description', 'O')]
Recording shape: (1, 1)

Structure field names:
Info
X
Y
Description


In [14]:
X = recording["X"][0, 0]

print("X type:", type(X))
print("X shape:", getattr(X, "shape", None))
print("X dtype:", getattr(X, "dtype", None))
print("X contents:\n", X)

X type: <class 'numpy.ndarray'>
X shape: (1, 3)
X dtype: [('Name', 'O'), ('Type', 'O'), ('Data', 'O'), ('Unit', 'O'), ('Raster', 'O')]
X contents:
 [[(array([], dtype='<U1'), array([[4]], dtype=int32), array([[-2.22416314e-04,  2.78020392e-05,  2.77592668e-04, ...,
           3.99938108e+00,  3.99963087e+00,  3.99988067e+00]],
        shape=(1, 16008)), array([], dtype='<U1'), array(['Mech_4kHz'], dtype='<U9'))
  (array([], dtype='<U1'), array([[4]], dtype=int32), array([[-3.42178944e-06,  1.19762630e-05,  2.78020392e-05, ...,
           3.99996878e+00,  3.99998460e+00,  4.00000000e+00]],
        shape=(1, 256823)), array([], dtype='<U1'), array(['HostService'], dtype='<U11'))
  (array([], dtype='<U1'), array([[4]], dtype=int32), array([[0.        , 0.99961419, 1.99922796, 2.99884215, 3.99839689]]), array([], dtype='<U1'), array(['Temp_1Hz'], dtype='<U8'))]]


In [15]:
X = recording["X"][0, 0]

for i in range(X.shape[1]):
    channel = X[0, i]

    print(f"\n--- Channel {i + 1} ---")

    print("Name:", channel["Name"])
    print("Type:", channel["Type"])
    print("Data shape:", channel["Data"].shape)
    print("Unit:", channel["Unit"])
    print("Raster:", channel["Raster"])

    data = channel["Data"].flatten()

    print("First 10 samples:", data[:10])
    print("Minimum:", data.min())
    print("Maximum:", data.max())


--- Channel 1 ---
Name: []
Type: [[4]]
Data shape: (1, 16008)
Unit: []
Raster: ['Mech_4kHz']
First 10 samples: [-2.22416314e-04  2.78020392e-05  2.77592668e-04  5.27383298e-04
  7.77173927e-04  1.02739228e-03  1.27718291e-03  1.52697354e-03
  1.77676417e-03  2.02698252e-03]
Minimum: -0.00022241631372343917
Maximum: 3.9998806650932135

--- Channel 2 ---
Name: []
Type: [[4]]
Data shape: (1, 256823)
Unit: []
Raster: ['HostService']
First 10 samples: [-3.42178944e-06  1.19762630e-05  2.78020392e-05  4.32000917e-05
  5.90258679e-05  7.44239204e-05  9.02496965e-05  1.05647749e-04
  1.21473525e-04  1.36871578e-04]
Minimum: -3.4217894419041883e-06
Maximum: 4.0

--- Channel 3 ---
Name: []
Type: [[4]]
Data shape: (1, 5)
Unit: []
Raster: ['Temp_1Hz']
First 10 samples: [0.         0.99961419 1.99922796 2.99884215 3.99839689]
Minimum: 0.0
Maximum: 3.998396891646471


In [16]:
Y = recording["Y"][0, 0]

print("Y type:", type(Y))
print("Y shape:", getattr(Y, "shape", None))
print("Y dtype:", getattr(Y, "dtype", None))
print("\nY contents:\n", Y)

Y type: <class 'numpy.ndarray'>
Y shape: (1, 7)
Y dtype: [('Name', 'O'), ('Type', 'O'), ('Data', 'O'), ('Unit', 'O'), ('Raster', 'O'), ('Device', 'O'), ('XIndex', 'O'), ('DownSampling', 'O'), ('Description', 'O'), ('DisplayIdentifier', 'O'), ('Path', 'O'), ('Flags', 'O'), ('Min', 'O'), ('Max', 'O'), ('MinWeak', 'O'), ('MaxWeak', 'O')]

Y contents:
 [[(array(['force'], dtype='<U5'), array([[4]], dtype=int32), array([[927.26279233, 919.93857359, 949.54062435, ..., 927.56796811,
          929.3990228 , 927.87314389]], shape=(1, 16008)), array([], dtype='<U1'), array(['Mech_4kHz'], dtype='<U9'), array(['Platform'], dtype='<U8'), array([[1]], dtype=int32), array([[1]], dtype=int32), array([], dtype='<U1'), array([], dtype='<U1'), array(['Model Root/Einggangssignal'], dtype='<U26'), array([[2]], dtype=int32), array([[-1.79769313e+308]]), array([[1.79769313e+308]]), array([[-1.79769313e+308]]), array([[1.79769313e+308]]))
  (array(['phase_current_1'], dtype='<U15'), array([[4]], dtype=int32),

In [19]:
import numpy as np

Y = recording["Y"][0, 0]

signals = {}

for i in range(Y.shape[1]):
    channel = Y[0, i]

    name = str(channel["Name"].item()[0])
    data = np.asarray(channel["Data"]).flatten()

    if name in [
        "vibration_1",
        "phase_current_1",
        "phase_current_2"
    ]:
        signals[name] = data

for name, data in signals.items():
    print(f"\n{name}")
    print("Number of samples:", len(data))
    print("Minimum:", data.min())
    print("Maximum:", data.max())